## Quack Elo Prototype Notebook

This is intended to be a prototype notebook. It is suposed to be implemented as a module / service which calculates a customized version of elo per player after each map, by accessing the database.

In [19]:
import pandas as pd
from db.connection_db import get_conn


def table_columns(conn, table_name: str) -> set[str]:
    rows = conn.execute(f"PRAGMA table_info({table_name})").fetchall()
    return {str(r[1]) for r in rows}


def first_existing(columns: set[str], candidates: list[str], label: str) -> str:
    for candidate in candidates:
        if candidate in columns:
            return candidate
    raise RuntimeError(f"Could not find a usable {label} column. Tried: {candidates}")


with get_conn() as conn:
    mps_cols = table_columns(conn, "match_player_stats")
    m_cols = table_columns(conn, "matches")

    steam_col = first_existing(mps_cols, ["steamid64", "steam64_id"], "steam id")
    name_col = first_existing(mps_cols, ["name", "player_name"], "player name")
    team_col = first_existing(mps_cols, ["team", "team_name"], "team")

    # 1) matchhistory table (match_id as unique key) sorted by date

    if "match_id" in m_cols:
        preferred_cols = [
            "match_id",
            "match_time",
            "start_time",
            "end_time",
            "created_at",
            "finished_at",
            "winner",
            "team1_score",
            "team2_score",
        ]
        match_cols = [c for c in preferred_cols if c in m_cols]
        matchhistory_df = pd.read_sql_query(
            f"SELECT {', '.join(match_cols)} FROM matches ORDER BY match_id DESC",
            conn,
        )
    else:
        matchhistory_df = pd.read_sql_query(
            """
            SELECT DISTINCT match_id
            FROM match_player_stats
            ORDER BY match_id DESC
            """,
            conn,
        )

    matchhistory_df = matchhistory_df.drop_duplicates(subset=["match_id"], keep="first")
    matchhistory_df = matchhistory_df.set_index("match_id", drop=False)
    # sort by date if possible
    date_cols = [c for c in ["match_time", "start_time", "end_time", "created_at", "finished_at"] if c in matchhistory_df.columns]
    if date_cols:
        matchhistory_df = matchhistory_df.sort_values(date_cols, ascending=False, kind="stable")

    # 2) playerlist table (teams per match)
    playerlist_df = pd.read_sql_query(
        f"""
        SELECT DISTINCT
            mps.match_id,
            CAST(mps.{steam_col} AS TEXT) AS steamid,
            TRIM(mps.{team_col}) AS team_name,
            TRIM(mps.{name_col}) AS player_name
        FROM match_player_stats mps
        WHERE mps.{steam_col} IS NOT NULL
          AND TRIM(COALESCE(mps.{team_col}, '')) <> ''
          AND TRIM(COALESCE(mps.{name_col}, '')) <> ''
        ORDER BY mps.match_id DESC, team_name, player_name
        """,
        conn,
    )
    playerlist_df = playerlist_df.drop_duplicates(
        subset=["match_id", "team_name", "steamid"],
        keep="first",
    )

    # 3) players table (distinct players)
    players_df = playerlist_df[["steamid", "player_name", "match_id"]].copy()
    players_df = players_df.sort_values(["steamid", "match_id"], ascending=[True, False])
    players_df = players_df.drop_duplicates(subset=["steamid"], keep="first")
    players_df = players_df.drop(columns=["match_id"]).rename(columns={"player_name": "name"})
    players_df = players_df.sort_values(["name", "steamid"], kind="stable").reset_index(drop=True)

print("Built independent tables:")
print(f"matchhistory_df rows={len(matchhistory_df):,} unique_match_id={matchhistory_df['match_id'].nunique():,}")
print(f"playerlist_df rows={len(playerlist_df):,} unique_match_id={playerlist_df['match_id'].nunique():,}")
print(f"players_df rows={len(players_df):,} unique_steamid={players_df['steamid'].nunique():,}")

print("\nmatchhistory_df preview:")
display(matchhistory_df)

print("\nplayerlist_df preview:")
display(playerlist_df)

print("\nplayers_df preview:")
display(players_df)

Built independent tables:
matchhistory_df rows=12 unique_match_id=12
playerlist_df rows=121 unique_match_id=12
players_df rows=23 unique_steamid=23

matchhistory_df preview:


,match_id,start_time,end_time,created_at,winner,team1_score,team2_score
match_id,,,,,,,
12,12,2026-04-06 00:16:36,2026-04-05T23:18:14,2026-04-06 16:17:17,TeamA,13,9
10,10,2026-04-05 23:29:37,2026-04-05T22:31:15,2026-04-06 16:17:17,TeamB,9,13
11,11,2026-04-05 22:57:57,NaN,2026-04-06 16:17:17,,0,1
9,9,2026-04-03T22:45:08,2026-04-03T22:45:08,2026-04-06 16:22:36,TeamA,13,8
8,8,2026-04-03T21:58:13,2026-04-03T21:58:13,2026-04-06 16:22:24,TeamB,10,13
7,7,2026-04-03T21:07:13,2026-04-03T21:07:13,2026-04-06 16:22:08,TeamA,13,9
6,6,2026-03-27T21:10:25,2026-03-27T21:10:25,2026-04-06 16:21:56,TeamB,5,13
5,5,2026-03-27T20:33:52,2026-03-27T20:33:52,2026-04-06 16:21:45,TeamB,4,13
4,4,2026-03-20 23:46:46,2026-03-20T22:48:10,2026-04-06 16:17:17,TeamB,1,13



playerlist_df preview:


,match_id,steamid,team_name,player_name
0,9,76561198051680113,TeamA,Captain Blake
1,9,76561198260260482,TeamA,Chicca —ฅ/ᐠ. ̫ .ᐟ\ฅ —
2,9,76561197970397706,TeamA,Dude
3,9,76561198045998047,TeamA,KeTaNoS
4,9,76561197987271352,TeamA,Tundra ツ
...,...,...,...,...
116,1,76561198962927846,TeamB,Amü
117,1,76561198051680113,TeamB,Captain Blake
118,1,76561198034161077,TeamB,Couchyyy1337
119,1,76561198045998047,TeamB,KeTaNoS



players_df preview:


,steamid,name
0,76561198010796900,-Zuria-
1,76561198962927846,Amü
2,76561198051680113,Captain Blake
3,76561198260260482,Chicca —ฅ/ᐠ. ̫ .ᐟ\ฅ —
4,76561198034161077,Couchyyy1337
5,76561198240903150,Dottersack
6,76561197970397706,Dude
7,76561198007257679,Inay
8,76561198045998047,KeTaNoS
9,76561198016875985,Luffy-Eyss
